In [1]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [2]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


10

In [3]:
# =======================================================================================
#
# BLOCK 3: PYTORCH SETUP & FULL DATA PREPARATION
#
# =======================================================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler

print(f"PyTorch version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# --- Custom PyTorch Dataset ---
class HousePriceDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        features = torch.tensor(self.features[idx], dtype=torch.float32)
        if self.labels is not None:
            labels = torch.tensor(self.labels[idx], dtype=torch.float32)
            return features, labels
        else:
            return features

# --- Data Scaling (The Most Important Step for NNs) ---
print("\nScaling features AND target for the Neural Network...")

# 1. Scale the input features (X)
feature_scaler = StandardScaler()
X_scaled = feature_scaler.fit_transform(X)
X_test_scaled = feature_scaler.transform(X_test)

# 2. Scale the target variable (y_true)
target_scaler = StandardScaler()
y_true_scaled = target_scaler.fit_transform(y_true.to_numpy().reshape(-1, 1))

print("PyTorch setup and full data scaling complete.")

PyTorch version: 2.7.1+cu126
Using device: cuda

Scaling features AND target for the Neural Network...
PyTorch setup and full data scaling complete.


In [4]:
# =======================================================================================
#
# BLOCK 4: DEFINE THE OPTIMIZED NEURAL NETWORK ARCHITECTURE
#
# =======================================================================================
class OptimizedNet(nn.Module):
    def __init__(self, input_shape, layer_sizes=[512, 256, 128, 64], dropout_rates=[0.5, 0.4, 0.3, 0.2]):
        super(OptimizedNet, self).__init__()
        
        layers = []
        in_features = input_shape
        for size, dropout in zip(layer_sizes, dropout_rates):
            # Linear layer
            linear = nn.Linear(in_features, size)
            # Kaiming He initialization is excellent for ReLU-like activations
            nn.init.kaiming_normal_(linear.weight, mode='fan_in', nonlinearity='relu')
            layers.append(linear)
            
            layers.append(nn.BatchNorm1d(size))
            layers.append(nn.SiLU()) # SiLU (or Swish) can sometimes outperform ReLU
            layers.append(nn.Dropout(dropout))
            in_features = size
            
        layers.append(nn.Linear(in_features, 1)) # Final output layer
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

print("Optimized Neural Network architecture defined successfully.")

Optimized Neural Network architecture defined successfully.


In [5]:
# =======================================================================================
#
# BLOCK 5 (CORRECTED): K-FOLD TRAINING OF THE OPTIMIZED NEURAL NETWORK
#
# =======================================================================================
from torch.optim.lr_scheduler import ReduceLROnPlateau

print("\n--- Starting K-Fold training for the optimized Neural Network ---")

# --- Training Configuration ---
EPOCHS = 500
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
PATIENCE = 30 # Increased patience for early stopping

# Initialize prediction arrays
oof_nn_preds = np.zeros(len(X))
test_nn_preds = np.zeros(len(X_test))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, grade_for_stratify)):
    print(f"\n" + "="*50)
    print(f"--- TRAINING FOLD {fold+1}/{N_SPLITS} ---")
    print("="*50)
    
    # --- Create Datasets and DataLoaders ---
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y_true_scaled[train_idx], y_true_scaled[val_idx]
    train_dataset = HousePriceDataset(X_train, y_train)
    val_dataset = HousePriceDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # --- Initialize Model, Loss, Optimizer, and Scheduler ---
    model = OptimizedNet(input_shape=X_train.shape[1]).to(device)
    loss_fn = nn.L1Loss() 
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    
    # --- THE FIX IS HERE: Removed the 'verbose' argument ---
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None

    # --- Training Loop ---
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = loss_fn(outputs, labels)
            train_loss += loss.item()
            loss.backward()
            optimizer.step()
        train_loss /= len(train_loader)
            
        # --- Validation Loop ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += loss_fn(outputs, labels).item()
        val_loss /= len(val_loader)
        
        scheduler.step(val_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.1e}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = model.state_dict()
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping triggered at epoch {epoch+1}. Best validation loss: {best_val_loss:.6f}")
            break
            
    # --- Generate Predictions for this fold ---
    model.load_state_dict(best_model_state)
    model.eval()
    
    val_preds_list = []
    with torch.no_grad():
        for features, _ in val_loader:
            features = features.to(device)
            val_preds_list.append(model(features).cpu().numpy())
    raw_oof_preds = np.concatenate(val_preds_list)
    oof_nn_preds[val_idx] = target_scaler.inverse_transform(raw_oof_preds).flatten()
    
    test_dataset = HousePriceDataset(X_test_scaled)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_preds_list = []
    with torch.no_grad():
        for features in test_loader:
            features = features.to(device)
            test_preds_list.append(model(features).cpu().numpy())
    raw_test_preds = np.concatenate(test_preds_list)
    test_nn_preds += target_scaler.inverse_transform(raw_test_preds).flatten() / N_SPLITS

# --- Final Evaluation ---
final_mean_rmse_nn = np.sqrt(mean_squared_error(y_true, oof_nn_preds))
print("\n--- Neural Network K-Fold Training Complete ---")
print(f"Final NN Mean Model OOF RMSE: ${final_mean_rmse_nn:,.2f}")


--- Starting K-Fold training for the optimized Neural Network ---

--- TRAINING FOLD 1/5 ---
Epoch 001 | Train Loss: 0.336140 | Val Loss: 0.232844 | LR: 1.0e-03
Epoch 010 | Train Loss: 0.215517 | Val Loss: 0.179119 | LR: 1.0e-03
Epoch 020 | Train Loss: 0.195939 | Val Loss: 0.163253 | LR: 1.0e-03
Epoch 030 | Train Loss: 0.189518 | Val Loss: 0.158757 | LR: 1.0e-03
Epoch 040 | Train Loss: 0.185658 | Val Loss: 0.157354 | LR: 1.0e-03
Epoch 050 | Train Loss: 0.182552 | Val Loss: 0.155578 | LR: 1.0e-03
Epoch 060 | Train Loss: 0.181843 | Val Loss: 0.154024 | LR: 1.0e-03
Epoch 070 | Train Loss: 0.178950 | Val Loss: 0.151604 | LR: 1.0e-03
Epoch 080 | Train Loss: 0.177152 | Val Loss: 0.150006 | LR: 1.0e-03
Epoch 090 | Train Loss: 0.177054 | Val Loss: 0.151191 | LR: 1.0e-03
Epoch 100 | Train Loss: 0.173865 | Val Loss: 0.148072 | LR: 1.0e-03
Epoch 110 | Train Loss: 0.173408 | Val Loss: 0.148310 | LR: 1.0e-03
Epoch 120 | Train Loss: 0.173120 | Val Loss: 0.145605 | LR: 1.0e-03
Epoch 130 | Train Loss

In [6]:
# =======================================================================================
#
# BLOCK 6: SAVE MODEL PREDICTIONS
#
# =======================================================================================
import os

# --- Define the path for saving predictions ---
PREDS_SAVE_PATH = 'NN_model_predictions/'
os.makedirs(PREDS_SAVE_PATH, exist_ok=True)
print(f"Prediction arrays will be saved in: {PREDS_SAVE_PATH}")

# --- Save the OOF and Test prediction arrays to .npy files ---
# We use .npy format because it's fast and efficient for numpy arrays.

try:
    # Save the Out-of-Fold predictions for the training set
    np.save(f'{PREDS_SAVE_PATH}oof_nn_preds.npy', oof_nn_preds)
    
    # Save the final averaged predictions for the test set
    np.save(f'{PREDS_SAVE_PATH}test_nn_preds.npy', test_nn_preds)
    
    print("\nOOF and Test predictions for the Neural Network saved successfully.")
    print("You can now load these files in your main ensemble notebook.")

except NameError:
    print("\nERROR: Could not find 'oof_nn_preds' or 'test_nn_preds'.")
    print("Please ensure the K-Fold training cell (Block 5) has been run successfully first.")

Prediction arrays will be saved in: NN_model_predictions/

OOF and Test predictions for the Neural Network saved successfully.
You can now load these files in your main ensemble notebook.
